# 03 — Feature Engineering

## Notebook Purpose

This notebook creates engineered features for machine learning models.

### Features Created
- Calendar features
- Seasonal indicators
- Weather buckets
- Lagged traffic variables
- Rolling averages
- Operational weather flags

These features are intended to improve prediction of daily traffic-related 911 call volume.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Load merged traffic + weather dataset

traffic_weather = pd.read_csv("../data/processed/traffic_weather_daily.csv")

traffic_weather.head()

,date,traffic_calls,time,temperature_2m_mean (°F),temperature_2m_min (°F),temperature_2m_max (°F),precipitation_sum (inch),rain_sum (inch),snowfall_sum (inch),precipitation_hours (h),wind_speed_10m_max (mp/h),wind_gusts_10m_max (mp/h)
0,2024-01-01,119,1/1/24,38.4,34.9,42.9,0.000,0.000,0.000,0,5.6,8.3
1,2024-01-02,138,1/2/24,42.8,38.1,45.9,0.154,0.154,0.000,9,9.9,16.3
2,2024-01-03,134,1/3/24,44.8,40.1,47.0,0.150,0.150,0.000,8,14.2,22.6
3,2024-01-04,140,1/4/24,44.9,40.2,48.6,0.193,0.193,0.000,14,13.5,22.8
4,2024-01-05,178,1/5/24,44.0,40.9,47.3,0.457,0.453,0.028,12,20.4,34.9


In [4]:
# Basic checks

traffic_weather.info()
traffic_weather.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 866 entries, 0 to 865
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       866 non-null    object 
 1   traffic_calls              866 non-null    int64  
 2   time                       866 non-null    object 
 3   temperature_2m_mean (°F)   866 non-null    float64
 4   temperature_2m_min (°F)    866 non-null    float64
 5   temperature_2m_max (°F)    866 non-null    float64
 6   precipitation_sum (inch)   866 non-null    float64
 7   rain_sum (inch)            866 non-null    float64
 8   snowfall_sum (inch)        866 non-null    float64
 9   precipitation_hours (h)    866 non-null    int64  
 10  wind_speed_10m_max (mp/h)  866 non-null    float64
 11  wind_gusts_10m_max (mp/h)  866 non-null    float64
dtypes: float64(8), int64(2), object(2)
memory usage: 81.3+ KB


date                         0
traffic_calls                0
time                         0
temperature_2m_mean (°F)     0
temperature_2m_min (°F)      0
temperature_2m_max (°F)      0
precipitation_sum (inch)     0
rain_sum (inch)              0
snowfall_sum (inch)          0
precipitation_hours (h)      0
wind_speed_10m_max (mp/h)    0
wind_gusts_10m_max (mp/h)    0
dtype: int64

In [5]:
# Convert date column

traffic_weather["date"] = pd.to_datetime(traffic_weather["date"])

In [6]:
# Calendar features

traffic_weather["year"] = traffic_weather["date"].dt.year
traffic_weather["month"] = traffic_weather["date"].dt.month
traffic_weather["day_of_week"] = traffic_weather["date"].dt.day_name()
traffic_weather["day_of_week_num"] = traffic_weather["date"].dt.dayofweek
traffic_weather["weekend"] = traffic_weather["day_of_week_num"].isin([5, 6])

In [7]:
# Season feature

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

traffic_weather["season"] = traffic_weather["month"].apply(get_season)

In [8]:
# Precipitation features

traffic_weather["rain_day"] = traffic_weather["precipitation_sum (inch)"] > 0

def precip_bucket(x):
    if x == 0:
        return "No Rain"
    elif x < 0.1:
        return "Light Rain"
    elif x < 0.5:
        return "Moderate Rain"
    else:
        return "Heavy Rain"

traffic_weather["precip_bucket"] = traffic_weather["precipitation_sum (inch)"].apply(precip_bucket)

In [9]:
# Temperature features

def temp_bucket(x):
    if x < 32:
        return "Freezing"
    elif x < 50:
        return "Cold"
    elif x < 70:
        return "Mild"
    else:
        return "Warm"

traffic_weather["temp_bucket"] = traffic_weather["temperature_2m_max (°F)"].apply(temp_bucket)

In [10]:
# Extreme weather flags

traffic_weather["freezing_day"] = traffic_weather["temperature_2m_min (°F)"] <= 32
traffic_weather["hot_day"] = traffic_weather["temperature_2m_max (°F)"] >= 85
traffic_weather["heavy_rain_day"] = traffic_weather["precipitation_sum (inch)"] >= 0.5

In [11]:
# Lag features

traffic_weather = traffic_weather.sort_values("date")

traffic_weather["traffic_calls_lag_1"] = traffic_weather["traffic_calls"].shift(1)
traffic_weather["precipitation_lag_1"] = traffic_weather["precipitation_sum (inch)"].shift(1)
traffic_weather["temperature_max_lag_1"] = traffic_weather["temperature_2m_max (°F)"].shift(1)

In [12]:
# Rolling average features

traffic_weather["traffic_calls_7_day_avg"] = (
    traffic_weather["traffic_calls"]
    .rolling(window=7)
    .mean()
)

traffic_weather["precipitation_3_day_total"] = (
    traffic_weather["precipitation_sum (inch)"]
    .rolling(window=3)
    .sum()
)

In [13]:
# Check engineered features

traffic_weather.head()

,date,traffic_calls,time,temperature_2m_mean (°F),temperature_2m_min (°F),temperature_2m_max (°F),precipitation_sum (inch),rain_sum (inch),snowfall_sum (inch),precipitation_hours (h),...,precip_bucket,temp_bucket,freezing_day,hot_day,heavy_rain_day,traffic_calls_lag_1,precipitation_lag_1,temperature_max_lag_1,traffic_calls_7_day_avg,precipitation_3_day_total
0,2024-01-01,119,1/1/24,38.4,34.9,42.9,0.000,0.000,0.000,0,...,No Rain,Cold,False,False,False,NaN,NaN,NaN,NaN,NaN
1,2024-01-02,138,1/2/24,42.8,38.1,45.9,0.154,0.154,0.000,9,...,Moderate Rain,Cold,False,False,False,119.0,0.000,42.9,NaN,NaN
2,2024-01-03,134,1/3/24,44.8,40.1,47.0,0.150,0.150,0.000,8,...,Moderate Rain,Cold,False,False,False,138.0,0.154,45.9,NaN,0.304
3,2024-01-04,140,1/4/24,44.9,40.2,48.6,0.193,0.193,0.000,14,...,Moderate Rain,Cold,False,False,False,134.0,0.150,47.0,NaN,0.497
4,2024-01-05,178,1/5/24,44.0,40.9,47.3,0.457,0.453,0.028,12,...,Moderate Rain,Cold,False,False,False,140.0,0.193,48.6,NaN,0.800


In [16]:
traffic_weather.columns

Index(['date', 'traffic_calls', 'time', 'temperature_2m_mean (°F)',
       'temperature_2m_min (°F)', 'temperature_2m_max (°F)',
       'precipitation_sum (inch)', 'rain_sum (inch)', 'snowfall_sum (inch)',
       'precipitation_hours (h)', 'wind_speed_10m_max (mp/h)',
       'wind_gusts_10m_max (mp/h)', 'year', 'month', 'day_of_week',
       'day_of_week_num', 'weekend', 'season', 'rain_day', 'precip_bucket',
       'temp_bucket', 'freezing_day', 'hot_day', 'heavy_rain_day',
       'traffic_calls_lag_1', 'precipitation_lag_1', 'temperature_max_lag_1',
       'traffic_calls_7_day_avg', 'precipitation_3_day_total'],
      dtype='object')

In [15]:
# Save final modeling dataset

traffic_weather.to_csv(
    "../data/processed/traffic_weather_features.csv",
    index=False
)

In [16]:
# Confirm save

final = pd.read_csv("../data/processed/traffic_weather_features.csv")
final.shape

(866, 29)